> Processing of the results from training (including *clusterization*)

In [ ]:
import pathlib

import numpy as np
import pandas as pd

from torch.utils.data import DataLoader
import hdbscan

import trapallada.data
import trapallada.util
import trapallada.core
import trapallada.plot
import trapallada.analysis

So that `import`ed Python modules are automatically reloaded when modified

In [ ]:
%load_ext autoreload
%autoreload 2

The base directory

In [ ]:
base_dir = trapallada.analysis.get_base_directory()
base_dir

# Input Files

Particular *sub*directory with the trained model. It should be a `str` so that parameters can be injected with papermill.

In [ ]:
# model_subdir = 'results/normalization/normalized_rate/vanilla_x_sd_015'
# model_subdir = 'results/normalization/normalized_rate/fft_vanilla_x_sd_00025'
# ===============
# model_subdir = 'results/normalization/clean/vanilla'
# model_subdir = 'results/normalization/count_rate/vanilla_x_sd_015'
# ===============
# model_subdir = "results/masked/vanilla/vanilla_x_sd_015"
# ===============
model_subdir = "results/cvae/min_count_rate_gti_with_background/vanilla_x_sd_015"
# ===============
# model_subdir = '/tmp/vanilla_x_sd_015/'
# ===============
# model_subdir = "results/multichannel/rates/fft_z_30"
# model_subdir = "results/multichannel/rates_large/fft_z_30"
model_subdir = "results/multichannel/rates_large_infs/fft_z_50"

Just in case

In [ ]:
model_dir = base_dir / model_subdir
# model_dir = pathlib.Path('/tmp') / 'normalization' / 'count_rate' / 'vanilla_x_sd_015'
display(model_dir)
assert model_dir.exists()

From the results directory,
- the `Dataset`,
- the headers of the corresponding files,
- the ML model,
- a `DataFrame` with results from training, 
- a `dict` with the parameters

In [ ]:
ds, positions_df, model, results_df, parameters = trapallada.analysis.load_results_dir(model_dir)

The directory with the full signals is inferred

In [ ]:
full_signals_dir = (pathlib.Path.home() / parameters['dataset']['files']['signals_file']).parent / 'full_signals'

Data loaders for both the training and validation sets

In [ ]:
_, (train_ds, validation_ds) = trapallada.data.dataloaders_from_dataset(
    ds,
    splitting_parameters=parameters['training']['train_test_split'] | dict(random_state=parameters['PRNG seeds']['training-validation splitting']),
    batch_size=parameters['training']['batch_size'], batching_random_state=parameters['PRNG seeds']['training dataloader'], return_datasets=True)

# Latent space projection

## 2D

The **full** dataset is encoded into the latent space (last columns in the resulting `DataFrame`)

In [ ]:
fulldataset_dl = DataLoader(ds, batch_size=128, shuffle=False)
proj_df = trapallada.util.make_projections_df(ds, fulldataset_dl, model.encoder)
proj_df.loc[train_ds[:][0], 'set'] = 'training'
proj_df.loc[validation_ds[:][0], 'set'] = 'validation'
proj_df

Embedded space is close to a standard Gaussian?

In [ ]:
proj_df.filter(like='z_').agg(['mean', 'std'])

### Clustering

Clustering using [HDBSCAN](https://github.com/scikit-learn-contrib/hdbscan) with pretty much every default parameter. See [this notebook](https://nbviewer.org/github/scikit-learn-contrib/hdbscan/blob/master/notebooks/How%20HDBSCAN%20Works.ipynb) for an explanation.

In [ ]:
def columns_clusters(like: str, min_cluster_size: int) -> pd.Series:

    selected_cols = proj_df.filter(like=like)

    clusterer = hdbscan.HDBSCAN(min_cluster_size=min_cluster_size)
    
    labels = clusterer.fit_predict(selected_cols)
    
    print(f'number of clusters: {len(np.unique(labels))}')
    
    return pd.Categorical(labels)

Clusting based on the **full** latent space

In [ ]:
cluster_in_z = 'cluster in z'
proj_df[cluster_in_z] = columns_clusters('z_', 10)

Clusting based on the tSNE projection

In [ ]:
cluster_in_projected_space = 'cluster in projected space'
proj_df[cluster_in_projected_space] = columns_clusters('tsne_', 80)
proj_df[cluster_in_projected_space].value_counts()

### Extracting positions

Information extracted from the *headers* of the FITs files

In [ ]:
positions_df

For the sake of convenience

In [ ]:
pos_columns = ['RA', 'DEC', 'CRA', 'CDEC']
tsne_columns = [f'tsne_{i}' for i in range(1, 3)]

*Projections* `DataFrame` extended with positions. Notice the indexes in `proj_df` are kept.

In [ ]:
proj_df_with_positions = proj_df.join(positions_df, on=['tar file', 'source'], how='left')
assert not proj_df_with_positions.isna().any().any()
proj_df_with_positions

The above `DataFrame` is to be shared with a collaborator, who will use it to query database (notice that, actually, `positions_df` would be enough)

In [ ]:
proj_df_with_positions[pos_columns + tsne_columns].to_csv('pos_and_proj.csv', index=False)

### SIMBAD (XMATCH) database

An external catalog is used to extract some *alleged* labels associated with spatial coordinates

In [ ]:
pd.set_option('display.max_columns', None)

catalog_file = base_dir / 'experiments' / 'Cross_Simbad_r3.5.csv'
assert catalog_file.exists()
catalog_df = pd.read_csv(catalog_file)
catalog_df

Some columns of interest

In [ ]:
catalog_columns_of_interest = ['main_type', 'other_types', 'main_id']

The number of *records* in the catalog is much larger than that of requested positions (*windows*), i.e., **each query returned several matches**.

In [ ]:
len(catalog_df), len(proj_df_with_positions)

In particular, the number of *matches* for every position is

In [ ]:
groupby_catalog = catalog_df.groupby(pos_columns)
groupby_catalog.size()

Out of every group, we keep only the first element (in the given order)

In [ ]:
first_match_catalog_df = groupby_catalog.first()
first_match_catalog_df

After picking the first match for every position, the number of *matched* positions (see the above number of rows) is lower than that of *requested* positions. Hence, the DB didn't return any result for some requested positions.

In [ ]:
len(positions_df)

Data used for training is matched against the catalog (indexes are still consistent with those in `proj_df`, and hence also with those in `ds`)

In [ ]:
match_columns_of_interes = catalog_columns_of_interest + pos_columns
ext_proj_df = pd.merge(
    proj_df_with_positions,
    first_match_catalog_df.reset_index()[match_columns_of_interes], # a few columns in the catalog DataFrame
    on=pos_columns, how='left'
)
ext_proj_df

Two new columns are added
- one just combines the *tar* and *FIT* file, which univocally determines the signal,
- the other yields the number of window *within* every (*tar* file, *FIT* file) combination

In [ ]:
ext_proj_df['# window'] = ext_proj_df.groupby(['tar file', 'source']).cumcount()
ext_proj_df['signal'] = ext_proj_df['tar file'].str.replace('.tar', '') + '/' + ext_proj_df['source'].str.replace('.FTZ', '')
ext_proj_df

We make sure all the windows within a given signal all have the same `main_type`

In [ ]:
for group_name, group_df in ext_proj_df.groupby('signal'):
    assert len(group_df['main_type'].unique()) == 1

Number of *unique* signals

In [ ]:
ext_proj_df['signal'].nunique()

The number of windows for each signal

In [ ]:
ext_proj_df.groupby('signal').size()

Distinct *types* of objects

In [ ]:
ext_proj_df['main_type'].unique()

Number os windows that lack a `main_type`

In [ ]:
main_type_is_na = ext_proj_df['main_type'].isna()
main_type_is_na.sum()

A column encompassing the whole latent space vector for every window.

In [ ]:
# ext_proj_df['z'] = ext_proj_df.filter(like='z_').values.tolist()

Resulting `DataFrame` is saved

In [ ]:
analysis_file = model_dir / 'analysis.parquet'
ext_proj_df.to_parquet(analysis_file)

Empty `main_type`s are set to `UNKNOWN`

In [ ]:
# ext_proj_df['main_type'].fillna('UNKNOWN', inplace=True)